In [9]:
!pip install streamlit pyngrok pandas numpy scikit-learn matplotlib


In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

st.title("💄 Cosmetic Sales Forecast App")

st.write("Upload your past sales data or use the sample dataset below 👇")

uploaded_file = st.file_uploader("Upload CSV file (optional)", type=["csv"])

if uploaded_file:
    df = pd.read_csv(uploaded_file)
else:
    data = {
        "Month": ["May 2025", "June 2025", "July 2025", "August 2025", "September 2025", "October 2025"],
        "Price": [450, 460, 470, 475, 480, 490],
        "Units_Sold": [320, 340, 380, 410, 395, 420],
        "Customer_Satisfaction": [4.2, 4.3, 4.5, 4.6, 4.4, 4.7]
    }
    df = pd.DataFrame(data)

st.subheader("📊 Past Data")
st.write(df)

df["Month_Number"] = np.arange(1, len(df) + 1)

X = df[["Month_Number", "Price", "Customer_Satisfaction"]]
y = df["Units_Sold"]
model = LinearRegression()
model.fit(X, y)

future_months = ["Nov 2025", "Dec 2025", "Jan 2026"]
future_price = [495, 500, 505]
future_satisfaction = [4.6, 4.7, 4.8]

future_df = pd.DataFrame({
    "Month": future_months,
    "Price": future_price,
    "Customer_Satisfaction": future_satisfaction,
    "Month_Number": np.arange(len(df) + 1, len(df) + 4)
})

future_df["Predicted_Sales"] = np.round(model.predict(future_df[["Month_Number", "Price", "Customer_Satisfaction"]]))

st.subheader("📈 Future Sales Forecast")
st.write(future_df[["Month", "Predicted_Sales"]])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df["Month"], df["Units_Sold"], marker='o', label="Past Sales")
ax.plot(future_df["Month"], future_df["Predicted_Sales"], marker='o', linestyle="--", color='orange', label="Predicted Sales")
ax.set_title("Cosmetic Sales Forecast")
ax.set_xlabel("Month")
ax.set_ylabel("Units Sold")
ax.legend()
ax.grid(True)
st.pyplot(fig)

Overwriting app.py


In [12]:
import time
import subprocess
from pyngrok import ngrok

# Configuration
STREAMLIT_PORT = 8503  # Incrementing port again for a fresh bind
NGROK_TOKEN = "34dPDiZlqXjXKiIjRXJxlf2SVyS_6hDwKfpW6GBqvuNPRgc2j"

print("--- CRITICAL CLEANUP STARTING ---")
# 1. Kill all system processes for ngrok and streamlit
subprocess.run(['pkill', '-9', 'ngrok'])
subprocess.run(['pkill', '-9', 'streamlit'])

# 2. Pyngrok specific cleanup
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
    ngrok.kill()
except:
    pass

time.sleep(5)
print("Cleanup complete. If ERR_NGROK_334 persists, please Restart Runtime.")

# 3. Initialize and Start
ngrok.set_auth_token(NGROK_TOKEN)

print(f"Starting Streamlit on port {STREAMLIT_PORT}...")
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', str(STREAMLIT_PORT)])

time.sleep(10)

try:
    public_url = ngrok.connect(STREAMLIT_PORT)
    print("--- DEPLOYMENT SUCCESSFUL ---")
    print(f"Public URL: {public_url}")
except Exception as e:
    print(f"Error: {e}")
    print("\n[ACTION REQUIRED]: Please click 'Runtime' -> 'Restart runtime' to fix this sticky ngrok session.")

--- CRITICAL CLEANUP STARTING ---
Cleanup complete. If ERR_NGROK_334 persists, please Restart Runtime.
Starting Streamlit on port 8503...


Error: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://tentless-syreeta-pentapodic.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


[ACTION REQUIRED]: Please click 'Runtime' -> 'Restart runtime' to fix this sticky ngrok session.


In [14]:
import time
import subprocess
from pyngrok import ngrok

# 1. System Cleanup
!pkill -9 ngrok
!pkill -9 streamlit
time.sleep(2)

# 2. Configuration
NGROK_TOKEN = "34dPDiZlqXjXKiIjRXJxlf2SVyS_6hDwKfpW6GBqvuNPRgc2j"
PORT = 8506
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Start Streamlit
print(f"Starting Streamlit on port {PORT}...")
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', str(PORT)])
time.sleep(5)

try:
    # 4. Connect using a RANDOM URL (no domain specified)
    # We use a unique name to ensure a fresh session
    public_url = ngrok.connect(PORT, proto="http", bind_tls=True)
    print("--- SUCCESS ---")
    print(f"Your app is live at: {public_url}")
except Exception as e:
    print(f"Failed to connect: {e}")
    print("\nIf it still mentions 'tentless-syreeta-pentapodic', go to https://dashboard.ngrok.com/cloud-edge/domains and delete or unbind that domain.")

Starting Streamlit on port 8506...
--- SUCCESS ---
Your app is live at: NgrokTunnel: "https://tentless-syreeta-pentapodic.ngrok-free.dev" -> "http://localhost:8506"


In [17]:
import time
import subprocess
from pyngrok import ngrok

# 1. Total Reset
print("Cleaning up existing processes...")
!pkill -9 ngrok
!pkill -9 streamlit
time.sleep(3)

# 2. Configuration
PORT = 8507
TOKEN = "34dPDiZlqXjXKiIjRXJxlf2SVyS_6hDwKfpW6GBqvuNPRgc2j"
ngrok.set_auth_token(TOKEN)

# 3. Launch Streamlit with CORS/XSRF fixes
print(f"Launching Streamlit on port {PORT}...")
# These flags are critical for fixing the 'Failed to fetch' / '0p' errors in Colab
cmd = [
    "streamlit", "run", "app.py",
    "--server.port", str(PORT),
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
]

with open("streamlit_logs.txt", "w") as f:
    subprocess.Popen(cmd, stdout=f, stderr=f)

# Wait for the server to bind
time.sleep(10)

# 4. Create Tunnel
try:
    # Clear cached tunnels
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)

    public_url = ngrok.connect(PORT)
    print("\n--- DEPLOYMENT SUCCESSFUL ---")
    print(f"App Link: {public_url}")
    print("\nNote: If the page shows a connection error, please refresh the browser tab.")
except Exception as e:
    print(f"Deployment Error: {e}")

Cleaning up existing processes...
Launching Streamlit on port 8507...

--- DEPLOYMENT SUCCESSFUL ---
App Link: NgrokTunnel: "https://tentless-syreeta-pentapodic.ngrok-free.dev" -> "http://localhost:8507"

Note: If the page shows a connection error, please refresh the browser tab.


In [11]:
import os
import subprocess

# Forcibly kill any process named 'ngrok' at the system level
print("Forcibly killing all ngrok system processes...")
subprocess.run(['pkill', '-9', 'ngrok'])

# Also kill any streamlit processes to be sure
subprocess.run(['pkill', '-9', 'streamlit'])

print("System processes cleared. If you still see ERR_NGROK_334 after running the startup cell again, please use 'Runtime > Restart runtime'.")

Forcibly killing all ngrok system processes...
System processes cleared. If you still see ERR_NGROK_334 after running the startup cell again, please use 'Runtime > Restart runtime'.
